Libraries

In [ ]:
!pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
-f https://data.pyg.org/whl/torch-2.9.0+cpu.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.9.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.6/669.6 kB 54.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.6/809.6 kB 56.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.2/304.2 kB 29.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.3 MB/s eta 0:00:00


Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
import numpy as np
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import add_self_loops
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.utils import add_self_loops
from torch_geometric.nn import GATv2Conv
from collections import defaultdict
import numpy as np
import torch
import random
from torch_geometric.nn import global_mean_pool
from torch_geometric.data import Batch

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/pyg_lib/libpyg.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_scatter_cpu.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_spmm_cpu.so
  import torch_geometric.typing


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


SEEDS

In [ ]:
seed = 99
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


Drive mount

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
test_cache = torch.load(
    "/content/drive/MyDrive/cache/test_cache_updated.pt",
    weights_only=False
)
val_cache = torch.load(
    "/content/drive/MyDrive/cache/val_cache_updated.pt",
    weights_only=False
)
train_cache = torch.load(
    "/content/drive/MyDrive/cache/train_cache_updated.pt",
    weights_only=False
)


EAGATPathEncoder takes a single extracted path graph (drug–protein–protein–…) and produces one fixed-size embedding.

In [ ]:
class EAGATPathEncoder(nn.Module):
    def __init__(self, hidden_dim, heads, input_dim, edge_dim=3):
        super().__init__()

        self.gat1 = GATv2Conv(
            in_channels=input_dim,
            out_channels=hidden_dim // heads,
            heads=heads,
            concat=True,
            edge_dim=edge_dim,
            dropout=0.2
        )
        self.gat2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            heads=1,
            concat=False,
            edge_dim=edge_dim,
            dropout=0.2

        )
        # role-aware gates
        self.drug_gate = nn.Linear(hidden_dim, 1)
        self.protein_gate = nn.Linear(hidden_dim, 1)

        self.fusion = nn.Sequential(nn.Linear(2*hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2))

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr

        # ===== GNN =====
        x = self.gat1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.gat2(x, edge_index, edge_attr)

        # ===== Split roles =====
        batch = data.batch

        drug_mask = data.node_role == 0
        prot_mask = data.node_role != 0

        drug_x = x[drug_mask]
        prot_x = x[prot_mask]

        drug_batch = batch[drug_mask]
        prot_batch = batch[prot_mask]
       #gating
        drug_w = torch.sigmoid(self.drug_gate(drug_x))
        prot_w = torch.sigmoid(self.protein_gate(prot_x))
       # weighted features
        drug_x = drug_x * drug_w
        prot_x = prot_x * prot_w
        # ===== pool per graph =====
        drug_repr = global_mean_pool(drug_x, drug_batch)
        prot_repr = global_mean_pool(prot_x, prot_batch)

        fusion = torch.cat([drug_repr, prot_repr], dim=-1)

        # ===== Final representation =====
        path_repr = self.fusion(fusion)
        return path_repr

Given:A drug–protein pair, A set of path graphs connecting them

The model must: Encode each path using the EA-GAT path encoder Aggregate multiple path embeddings in a learnable way, Output one scalar interaction score

In [ ]:
class PathLevelModel(nn.Module):
    def __init__(self, path_encoder, hidden_dim):
        super().__init__()

        self.path_encoder = path_encoder

        self.path_attn = nn.Linear(hidden_dim, 1)

        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1)
        )

        self.null_embedding = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, samples):

        all_graphs = []
        counts = []

        # ---------- collect graphs from the whole batch ----------
        for sample in samples:

            graphs = random.sample(
                sample["graphs"],
                min(15, len(sample["graphs"]))
            )

            all_graphs.extend(graphs)
            counts.append(len(graphs))
        # ---------- no graphs ----------
        if len(all_graphs) == 0:
            return self.scorer(
                self.null_embedding.unsqueeze(0)
            ).view(-1)
        # ---------- encode all paths together ----------

        batch_graph = Batch.from_data_list(all_graphs)

        batch_graph = batch_graph.to(self.null_embedding.device)


        z = self.path_encoder(batch_graph)


        z = torch.nan_to_num(z)

        # ---------- aggregate paths for every pair ----------
        outputs = []

        start = 0

        for n_paths in counts:

            if n_paths == 0:

                pair_embedding = self.null_embedding

            else:

                pair_z = z[start:start + n_paths]

                att_scores = self.path_attn(pair_z)
                att_weights = torch.softmax(att_scores, dim=0)

                pair_embedding = (att_weights * pair_z).sum(dim=0)

            outputs.append(pair_embedding)
            start += n_paths

        outputs = torch.stack(outputs, dim=0)

        logits = self.scorer(outputs)

        del batch_graph
        del z

        return logits.squeeze(-1)

In [ ]:
train_pairs = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/train_pairs_random.csv")
val_pairs  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/val_pairs_random.csv")
test_pairs = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/test_pairs_random.csv")

In [ ]:
train_pair_labels = {
    (row["drugbank_id"], row["uniprot_id"]): row["label"]
    for _, row in train_pairs.iterrows()
}

In [ ]:
val_pair_labels = {
    (row["drugbank_id"], row["uniprot_id"]): row["label"]
    for _, row in val_pairs.iterrows()
}

In [ ]:
test_pair_labels = {
    (row["drugbank_id"], row["uniprot_id"]): row["label"]
    for _, row in test_pairs.iterrows()
}

Group paths into pairs
ech row =  one path ===>each sample = one(drug, protein) with multiple paths

In [ ]:
from collections import defaultdict

def build_dataset(cache, pair_labels):

    grouped = defaultdict(list)

    for _, item in cache.items():
        key = (item["drug"], item["target"])
        grouped[key].append(item["graph"])

    dataset = []

    for key, graphs in grouped.items():

        if key not in pair_labels:
            continue

        dataset.append({

            "graphs": graphs,
            "num_paths": len(graphs),
            "label": float(pair_labels[key]),
            "drug": key[0],
            "target": key[1]

        })

    return dataset

In [ ]:
train_dataset = build_dataset(train_cache, train_pair_labels)
val_dataset   = build_dataset(val_cache, val_pair_labels)
test_dataset  = build_dataset(test_cache, test_pair_labels)

Dataset → batches (PyTorch-style)

In [ ]:
def batch_generator(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]


In [ ]:
path_encoder = EAGATPathEncoder(
    hidden_dim=128,
    heads=4,
    input_dim=256,
    edge_dim=3
).to(device)

In [ ]:
model = PathLevelModel(
    path_encoder=path_encoder,
    hidden_dim=128
).to(device)

optimizer

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


In [ ]:
criterion = nn.BCEWithLogitsLoss()


early stop

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=1e-4, mode='max', save_path='best_model.pt'):
        """
        patience: number of epochs to wait
        delta: minimum improvement to qualify as better
        mode: 'max' for AUC/accuracy, 'min' for loss
        """
        self.patience = patience
        self.delta = delta
        self.mode = mode
        self.save_path = save_path

        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, metric, model):
        score = metric if self.mode == 'max' else -metric

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)

        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.save_path)

In [ ]:
early_stopper = EarlyStopping(
    patience=10,
    mode='max',   # because you use AUC or accuracy
    save_path='best_model.pt'
)

In [ ]:
from sklearn.metrics import roc_auc_score
import random

num_epochs = 100
batch_size = 32
best_auc = 0
batch_counter = 0

for epoch in range(num_epochs):

    # ======================================================
    # TRAIN
    # ======================================================
    model.train()

    train_loss = 0.0
    train_true = []
    train_score = []
    train_samples = 0

    for i, batch in enumerate(batch_generator(train_dataset, batch_size)):
        batch_counter += 1
        optimizer.zero_grad()
        logits = model(batch)

        labels = torch.tensor(
            [sample["label"] for sample in batch],
            dtype=torch.float32,
            device=device
        )

        valid = ~(torch.isnan(logits) | torch.isinf(logits))
        if valid.sum() == 0:
            continue

        logits = logits[valid]
        labels = labels[valid]

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(labels)
        train_samples += len(labels)

        probs = torch.sigmoid(logits).detach().cpu().numpy()

        train_score.extend(probs.tolist())
        train_true.extend(labels.detach().cpu().numpy().tolist())

    train_loss /= max(train_samples, 1)
    train_auc = roc_auc_score(train_true, train_score)
    # ======================================================
    # VALIDATION
    # ======================================================
    model.eval()

    val_loss = 0.0
    val_samples = 0

    val_true = []
    val_score = []

    with torch.no_grad():

        for i, batch in enumerate(batch_generator(val_dataset, batch_size)):

            logits = model(batch)

            labels = torch.tensor(
                [sample["label"] for sample in batch],
                dtype=torch.float32,
                device=device
            )

            valid = ~(torch.isnan(logits) | torch.isinf(logits))

            if valid.sum() == 0:
                continue

            logits = logits[valid]
            labels = labels[valid]

            loss = criterion(logits, labels)

            val_loss += loss.item() * len(labels)
            val_samples += len(labels)

            probs = torch.sigmoid(logits).cpu().numpy()

            val_score.extend(probs.tolist())
            val_true.extend(labels.cpu().numpy().tolist())

    val_loss /= max(val_samples, 1)
    val_auc = roc_auc_score(val_true, val_score)

    # ======================================================
    # SAVE BEST MODEL
    # ======================================================
    best_auc = max(best_auc, val_auc)

    early_stopper(val_auc, model)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train AUC {train_auc:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val AUC {val_auc:.4f} | "
        f"Best {best_auc:.4f}"
    )

    if early_stopper.early_stop:
        print("Early stopping triggered.")
        break


model.load_state_dict(torch.load("best_model.pt", map_location=device))

print("Training Finished")
print("Best Validation AUC:", best_auc)


Epoch 001 | Train Loss 0.6441 | Train AUC 0.6154 | Val Loss 0.5482 | Val AUC 0.8502 | Best 0.8502
Epoch 002 | Train Loss 0.4487 | Train AUC 0.8618 | Val Loss 0.3837 | Val AUC 0.9125 | Best 0.9125
Epoch 003 | Train Loss 0.3517 | Train AUC 0.9119 | Val Loss 0.3579 | Val AUC 0.9324 | Best 0.9324
Epoch 004 | Train Loss 0.3131 | Train AUC 0.9299 | Val Loss 0.3222 | Val AUC 0.9495 | Best 0.9495
Epoch 005 | Train Loss 0.2857 | Train AUC 0.9407 | Val Loss 0.3137 | Val AUC 0.9577 | Best 0.9577
Epoch 006 | Train Loss 0.2702 | Train AUC 0.9454 | Val Loss 0.3074 | Val AUC 0.9638 | Best 0.9638
Epoch 007 | Train Loss 0.2565 | Train AUC 0.9500 | Val Loss 0.2754 | Val AUC 0.9659 | Best 0.9659
Epoch 008 | Train Loss 0.2449 | Train AUC 0.9554 | Val Loss 0.2870 | Val AUC 0.9681 | Best 0.9681
Epoch 009 | Train Loss 0.2411 | Train AUC 0.9552 | Val Loss 0.2989 | Val AUC 0.9699 | Best 0.9699
Epoch 010 | Train Loss 0.2298 | Train AUC 0.9595 | Val Loss 0.3256 | Val AUC 0.9729 | Best 0.9729
EarlyStopping counte

In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))

<All keys matched successfully>

picking threshold

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.linspace(0.1, 0.9, 81)
f1s = [
    f1_score(
        val_true,
        (np.array(val_score) >= t).astype(int)
    )
    for t in thresholds
]

best_threshold = thresholds[np.argmax(f1s)]
print("Selected validation threshold:", best_threshold)


Selected validation threshold: 0.37


test

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

model.eval()

y_true = []
y_score = []

with torch.no_grad():

    for batch in batch_generator(test_dataset, batch_size):

        for sample in batch:

            # -------- retrieve graphs --------
            drug = sample["drug"]
            protein = sample["target"]
            label = sample["label"]

            # -------- forward (PAIR LEVEL) --------
            logit = model([sample])          # scalar
            prob = torch.sigmoid(logit).item()    # probability

            y_true.append(label)
            y_score.append(prob)


In [ ]:
if len(y_true) == 0:
    print("No valid evaluation samples.")


roc_auc = roc_auc_score(y_true, y_score)
pr_auc = average_precision_score(y_true, y_score)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")


ROC-AUC: 0.9883
PR-AUC:  0.9831


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)
y_pred = (np.array(y_score) >= best_threshold).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)

print(f"Accuracy:            {acc:.4f}")
print(f"Precision:           {prec:.4f}")
print(f"Recall:              {rec:.4f}")
print(f"F1-score:            {f1:.4f}")
print(f"Balanced Accuracy:   {bal_acc:.4f}")


Accuracy:            0.9514
Precision:           0.9483
Recall:              0.9233
F1-score:            0.9356
Balanced Accuracy:   0.9460


In [ ]:
cm = confusion_matrix(y_true, y_pred)

cm_percent = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

df_cm = pd.DataFrame(
    np.round(cm_percent, 2),
    index=["True Negative", "True Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

print(df_cm)


               Predicted Negative  Predicted Positive
True Negative               96.88                3.12
True Positive                7.67               92.33
